In [37]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
import time

2026-02-24 12:05:16.410370694 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


### Embedding Model

In [38]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
model.get_sentence_embedding_dimension()

384

### Qdrant Create Collection

In [40]:
collection_name = "qdrant-documents"
vector_name = "all-MiniLM-L6-v2"

In [41]:
client = QdrantClient("http://localhost:6333")

In [42]:
client.delete_collection(collection_name)

False

In [43]:
client.create_collection(
    collection_name,
    vectors_config={
        vector_name: models.VectorParams(
            size=model.get_sentence_embedding_dimension(),
            distance=models.Distance.COSINE,
        ),       
    }
)

True

### Insert Document

In [32]:
documents = pd.read_csv("documents.csv")

In [33]:
len(documents)

200

In [34]:
documents["category"].value_counts()

category
qdrant.usecase            18
qdrant.performance        18
qdrant.clustering         16
qdrant.search             15
qdrant.configuration      13
qdrant.monitoring         12
qdrant.integration        11
qdrant.security           10
qdrant.storage            10
qdrant.payload             9
qdrant.collections         9
qdrant.vectors             9
qdrant.operations          9
qdrant.backup              7
qdrant.troubleshooting     6
qdrant.deployment          5
qdrant.comparison          5
qdrant.migration           5
qdrant.api                 4
qdrant.scaling             4
qdrant.maintenance         3
qdrant.setup               2
Name: count, dtype: int64

In [35]:
documents.head()

,query,category
0,install qdrant using docker on linux server,qdrant.setup
1,set up qdrant cloud cluster step by step,qdrant.deployment
2,create a new collection with cosine distance i...,qdrant.collections
3,configure vector size when creating qdrant col...,qdrant.collections
4,insert single vector point into qdrant,qdrant.vectors


In [44]:
list_of_queries = documents["query"].tolist()

In [45]:
len(list_of_queries)

200

In [46]:
list_of_queries[0]

'install qdrant using docker on linux server'

In [47]:
client.upsert(
    collection_name,
    points=[
        models.PointStruct(
            id=i,
            vector={
                vector_name: model.encode([document])[0],
            },
            payload={"text": document},
        )
        for i, document in enumerate(list_of_queries, start=1)
    ],
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

### Query

In [48]:
def dense_query(q: str, limit: int = 5) -> list[dict]:
    start_time = time.monotonic()
    embedding = model.encode([q])[0]
    end_time = time.monotonic()
    
    print("Dense vector generation time:", end_time - start_time)

    start_time = time.monotonic()
    result = client.query_points(
        collection_name,
        query=embedding,
        using=vector_name,
        limit=limit,
        with_payload=True,
    )
    end_time = time.monotonic()
    print("Query time:", end_time - start_time)

    return [point.payload for point in result.points]    

In [49]:
query = "search performance in Qdrant"

In [50]:
dense_hits = dense_query(query)

Dense vector generation time: 0.01928854900143051
Query time: 0.018377080999925965


In [51]:
dense_hits

[{'text': 'optimize search accuracy vs speed tradeoff in qdrant'},
 {'text': 'increase search throughput in qdrant'},
 {'text': 'diagnose slow query performance in qdrant'},
 {'text': 'search with score threshold in qdrant'},
 {'text': 'tune search batch size in qdrant'}]

### Test Queries

In [7]:
test_queries = pd.read_csv("test_queries.csv")

In [9]:
test_queries.head()

,query,desired_category
0,how to install qdrant on ubuntu server,qdrant.setup
1,running qdrant with docker compose example,qdrant.setup
2,qdrant cloud vs self hosted comparison,qdrant.deployment
3,how to create collection in qdrant using pytho...,qdrant.collections
4,qdrant collection configuration vector size an...,qdrant.collections


In [10]:
len(test_queries)

50

In [11]:
test_queries["desired_category"].value_counts()

desired_category
qdrant.search             6
qdrant.performance        5
qdrant.integration        4
qdrant.vectors            3
qdrant.clustering         3
qdrant.api                3
qdrant.usecase            3
qdrant.operations         3
qdrant.setup              2
qdrant.deployment         2
qdrant.collections        2
qdrant.payload            2
qdrant.scaling            2
qdrant.security           2
qdrant.backup             1
qdrant.monitoring         1
qdrant.migration          1
qdrant.storage            1
qdrant.comparison         1
qdrant.maintenance        1
qdrant.configuration      1
qdrant.troubleshooting    1
Name: count, dtype: int64

### Compute Metrics

In [ ]:
def compute_metrics(queries, top_k=5):
    def preprocess_text(text):
        text = text.strip()
        return text

    results = []
    for item in queries:
        query = item["query"]
        desired_category = item["desired_category"]  
        
        

In [4]:
k_values = [5, 20, 50]

for k in k_values:
    print(f"\n{'='*80}")
    print(f"Results with K={k}:")
    print('='*80)
    
    # results = compute_metrics(test_queries, top_k=k)
    # for result in results:
    #     print(f"Query: {result['query']}")
    #     print(f"  Precision@{k}: {result['precision@k']:.2f}, Recall@{k}: {result['recall@k']:.2f}")
    #     print()    


Results with K=5:

Results with K=20:

Results with K=50:
